In [1]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
from sklearn.metrics import classification_report, accuracy_score

# Path based on your sibling folder structure
TRAIN_PATH = "../EMBER/train_ember_2018_v2_features.parquet"
TEST_PATH = "../EMBER/test_ember_2018_v2_features.parquet"

In [4]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

# Paths as established previously
TRAIN_PATH = "../EMBER/train_ember_2018_v2_features.parquet"
TEST_PATH = "../EMBER/test_ember_2018_v2_features.parquet"

print("Loading EMBER dataset with ultimate memory-safe sampling...")

def load_safe_sample(path, sample_size=10000):
    # Step 1: Read only the first row to check column names without loading the file
    first_row = pd.read_parquet(path, engine='pyarrow').iloc[:1]
    
    # Standardize label column name (checks for 'label' or 'Label')
    col_map = {col: col.lower() for col in first_row.columns}
    label_col = next((c for c in first_row.columns if c.lower() == 'label'), None)
    
    if not label_col:
        raise ValueError(f"Could not find a label column in {path}")

    # Step 2: Use PyArrow to get total row count
    table_meta = pq.read_metadata(path)
    total_rows = table_meta.num_rows
    print(f"File {path.split('/')[-1]} has {total_rows} total rows.")

    # Step 3: Load the sample
    # We load a bit more than needed then sample, to ensure we get a good mix
    df = pd.read_parquet(path, engine='pyarrow').sample(n=sample_size, random_state=42)
    
    # Normalize the label column to lowercase 'label' for the rest of the code
    df = df.rename(columns={label_col: 'label'})
    return df

try:
    # Reducing sample size to 10,000 as suggested by your error output
    train_df = load_safe_sample(TRAIN_PATH, sample_size=10000)
    test_df = load_safe_sample(TEST_PATH, sample_size=5000)
    
    # Filter out unlabeled data (-1) as per EMBER documentation [cite: 214, 216]
    train_df = train_df[train_df['label'] != -1]
    test_df = test_df[test_df['label'] != -1]

    print(f"✅ Success! Training Shape: {train_df.shape}")
    print(f"✅ Success! Testing Shape: {test_df.shape}")
    
except Exception as e:
    print(f"❌ Error during load: {e}")

Loading EMBER dataset with ultimate memory-safe sampling...
File train_ember_2018_v2_features.parquet has 799912 total rows.
❌ Error during load: malloc of size 7621561536 failed


In [3]:
# Separate features and labels
X_train = train_df.drop(columns=['label', 'sha256'], errors='ignore')
y_train = train_df['label']

X_test = test_df.drop(columns=['label', 'sha256'], errors='ignore')
y_test = test_df['label']

# EMBER features are often broad; we keep only numeric ones for LightGBM
X_train = X_train.select_dtypes(include=[np.number])
X_test = X_test.select_dtypes(include=[np.number])

print(f"Features ready. Training on {X_train.shape[1]} features.")

KeyError: 'label'

In [ ]:
import lightgbm as lgb
from sklearn.metrics import classification_report

# Initialize LightGBM with binary objective for Benign vs Malicious
ember_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=12,
    num_leaves=31,
    objective='binary',
    class_weight='balanced',
    random_state=42
)

print("Starting EMBER Model B training...")
ember_model.fit(X_train, y_train)
print("Training Complete.")

# Evaluate against known malware signatures
y_pred = ember_model.predict(X_test)
print("\n--- EMBER Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malicious']))

In [ ]:
import joblib

# Calculate threat probability (Model B output)
# Probability of class 1 (Malicious)
ember_threat_scores = ember_model.predict_proba(X_test)[:, 1]

print(f"Sample File Threat Scores: {ember_threat_scores[:5]}")

# Export the package for the Agent's telemetry collection layer
export_ember = {
    "model": ember_model,
    "features": X_train.columns.tolist(),
    "version": "1.0-EMBER",
    "metadata": "Supervised File Classifier for AEGIS Layer 1"
}

joblib.dump(export_ember, "aegis_ember_model.pkl")
print("\n✅ aegis_ember_model.pkl saved.")

In [ ]:
# This is how the EDR Agent will eventually process the scores locally:
# Final Score = (Model_A_Anomaly_Score * 0.5) + (Model_B_Classification_Score * 0.5)

def get_final_verdict(final_score):
    if final_score > 0.80:
        return "CRITICAL - Full Network Isolation" # [cite: 71]
    elif final_score > 0.60:
        return "HIGH - Kill Process & Quarantine" # [cite: 71]
    elif final_score > 0.30:
        return "MEDIUM - Initiate Peer Voting Round" # [cite: 43, 71]
    else:
        return "LOW - Log and Notify" # [cite: 71]